# Training and Testing

In [ ]:
from run_stock_factor import StockFactor
from langchain_openai import ChatOpenAI
import os
from datetime import datetime, timedelta

llm = ChatOpenAI(
  openai_api_key = os.getenv('OPENAI_API_KEY'),
  model='gpt-4o-mini',
  temperature=1,
)

# 買進賣出需要的手續費
fee = {
  'tax_stock' : 0.003,          # 買進與賣出各0.003
  'tax_future' : 0.00002 * 2,   # 買進與賣出各0.00002
  'fee_stock' : 0.001425 * 2,   # 券商手續費公道價，買進與賣出各0.001425
  'fee_future' : 0.00002 * 2,   # 個家不大相同，以交易稅計算
  'sliding_price' : 0.00001,    # 滑價
}

data = [
    ["2330", "台積電", "tw"],
    ["2317", "鴻海", "tw"],
    ["2454", "聯發科", "tw"],
    ["2382", "廣達", "tw"],
    ["3231", "緯創", "tw"],
    # ["2324","仁寶", "tw"],
    # ["4938", "和碩", "tw"],
    # ["2356","英業達", "tw"],
    # ["2881", "富邦金", "tw"],
    # ["2882", "國泰金", "tw"],
    # ["2412", "中華電", "tw"],
    # ["3045", "台灣大", "tw"],
    # ["6505", "台塑化", "tw"],
    # ["2603", "長榮", "tw"],
    
    ["AAPL", "Apple Inc.", "us"],
    ["GOOGL", "google", "us"],
    ["MSFT", "microsoft", "us"],
    ["AMZN", "amazon inc.", "us"],
    ["TSLA", "tesla", "us"],
    ["NVDA", "nvidia", "us"],
    # ["META", "Meta Platforms", "us"],
    # ["BRK", "Berkshire Hathaway", "us"],
]

for stock in data:
    env = {
        "stock_id" : stock[0],
        "stock_name" : stock[1],
        "country" : stock[2],
        "start_date" : "20230401",
        "end_date" : "20240331",
    }
    print(env['stock_id'], env['stock_name'], env['country'])

    stock_factor = StockFactor(llm, env)
    stock_factor.run_factors_expand()
    train_datarange, test_datarange = stock_factor.split_expand()

    mode = 0
    stock_factor.run_taining(mode, train_datarange, test_datarange)
    mode = 1
    stock_factor.run_taining(mode, train_datarange, test_datarange)



## 顯示所有

In [5]:
import os
import json
import pandas as pd

env = {
    "stock_id" : stock[0],
    "stock_name" : stock[1],
    "country" : stock[2],
    "start_date" : "20230401",
    "end_date" : "20240331",
}
path_out = "out_stock/GA_factor_" + env['start_date'] + "_" + env['end_date'] + "/"

folders = os.listdir(path_out)
results = []
index = []

for folder in folders:
    if folder == "factors_eng.json" or folder == "factors.json" or folder == "old_expand" or folder == ".DS_Store":
        continue
    folders_result = os.listdir(f"{path_out}{folder}")
    # print(f"{folder}:")
    for file in folders_result:
        if file == "factors.json" or file == "factors1.json" or file == "factors2.json" or file == "expand.json" or file == "expand1.json" or file == "expand2.json":
            continue
        path = f"{path_out}{folder}/{file}/result.json"
        with open(path, 'r') as f:
            json_data = json.load(f)
        # print(json_data)
        results.append(json_data['test'])
        index.append(f"{folder}_{file}")


# print(f"共有 {len(results), len(index)} 筆資料")
# print(results)

df = pd.DataFrame(results, index=index)
selected_columns = ['accuracy', 'precision', 'ev', 'precision_ev',
                    'avail_count', 'tp', 'fp', 'tn', 'fn']  # 替換為您需要的欄位名稱
df['accuracy'] = df['accuracy'].apply(lambda x: f"{x * 100:.2f}%")
df['precision'] = df['precision'].apply(lambda x: f"{x * 100:.2f}%")
df_selected = df.loc[:, selected_columns]
df_sorted = df_selected.sort_index()
df_sorted

,accuracy,precision,ev,precision_ev,avail_count,tp,fp,tn,fn
2317_ac,43.64%,44.23%,0.003220,0.003440,55,23,29,1,2
2317_ev,50.00%,63.64%,0.004074,0.016029,54,7,4,20,23
2330_ac,43.40%,40.82%,0.000425,-0.000052,53,20,29,3,1
2330_ev,53.19%,48.72%,0.001353,0.000509,47,19,20,6,2
2382_ac,71.43%,75.00%,0.014452,0.023194,7,3,1,2,1
2382_ev,69.57%,71.43%,0.008393,0.012285,23,5,2,11,5
2454_ac,51.16%,39.13%,0.002271,-0.002241,43,9,14,13,7
2454_ev,51.79%,45.00%,0.005182,0.003171,56,18,22,11,5
3231_ac,42.31%,42.86%,-0.001132,-0.002696,26,9,12,2,3
3231_ev,33.93%,34.88%,-0.006273,-0.006637,56,15,28,4,9


In [ ]:
# super training
# from api import genetic_algorithm
# from api import eval
# import os
# import shutil

# def run(mode, path):
#     # setting --------------------------------
#     mode = mode     # 0: 準確率, 1: 期望值
#     out_folder = path
#     # ----------------------------------------

#     state_file= out_folder + '/state.json'
#     file_gen= out_folder + '/generation_results.json'
#     file_result= out_folder + '/result.json'

#     print("Start, 第一次跑的話請確認state是空的")
#     best_individual = genetic_algorithm(factors, population_size=20, generations=50, state_file=state_file, results_file=file_gen, env=env, mode=mode, datarange=train_datarange)
#     print(f"Best individual: {best_individual}")

#     individual = best_individual
#     res_train = eval(individual, train_datarange, env)
#     res_test = eval(individual, test_datarange, env)
#     res = {
#         "train": res_train,
#         "test": res_test,
#         "individual": individual,
#     }

#     old_out_folder = out_folder + '1'
#     old_file_result= out_folder + '1/result.json'
#     with open(old_file_result, 'r') as f:
#         old_res = json.load(f)

#     if res['test']['ev'] > old_res['test']['ev']:
#         print("New result is better, update the file")
#         with open(file_result, 'w') as f:
#             json.dump(res, f, ensure_ascii=False, indent=4)
#         if os.path.isdir(old_out_folder):
#             shutil.rmtree(old_out_folder)
#         elif os.path.isfile(old_out_folder):
#             os.remove(old_out_folder)
#         # rename the file
#         os.rename(out_folder, out_folder + '1')
#     else:
#         print(f"New result is not better, keep the old result, old res: {old_res['test']['ev']}, new: {res['test']['ev']}")
#         # remove the directory or file safely
#         if os.path.isdir(out_folder):
#             shutil.rmtree(out_folder)
#         elif os.path.isfile(out_folder):
#             os.remove(out_folder)


# for i in range(1000):
#     run(0, f"{env['path_out']}/{env['stock_id']}/ac")
#     run(1, f"{env['path_out']}/{env['stock_id']}/ev")

In [ ]:
# plot
path_price_his = f"{os.path.dirname(os.path.abspath(os.getcwd()))}/history_data/{env['country']}/stock_price/{env['stock_id']}.csv"
df_price_his = pd.read_csv(path_price_his, encoding='utf-8')
df_price_his['Date'] = pd.to_datetime(df_price_his['Date'], format='%Y%m%d')
list_bnh_rtn = []
list_factor_rtn = []
balance = df_price_his[df_price_his['Date'].dt.strftime(
    '%Y%m%d') == res_test['rtn_list'][0][0]]['Open'].values[0]
list_date = []


for date, rtn in res_test['rtn_list']:
    # print(rtn)
    balance = balance * (1 + float(rtn))
    list_date.append(date)
    list_bnh_rtn.append(df_price_his[df_price_his['Date'].dt.strftime(
        '%Y%m%d') == date]['Close'].values[0])
    list_factor_rtn.append(balance)

print(list_date)
print(list_bnh_rtn)
print(list_factor_rtn)
print(len(list_date))
print(len(list_bnh_rtn))
print(len(list_factor_rtn))

df = pd.DataFrame(list_factor_rtn, index=list_date, columns=['balance'])
dplot = [datetime.strptime(d, '%Y%m%d').date() for d in list_date]


plt.title(f" {env['stock_id']} trading results")
plt.plot(dplot, list_bnh_rtn, label="buy and hold")  # blue
plt.plot(dplot, list_factor_rtn, label="factor trading")  # orange
plt.legend()